1. Replace values in one call

Using df_movies, in the "country" column replace three values in a single .replace() call:
"American"→"USA", "Korean"→"ROK", and "Thailand"→"THA".

Do not modify any other columns.

In [2]:
import pandas as pd
import numpy as np

In [3]:
titles = ["Past Lives", "Citizen Kane", "Revenge", "Okja", "Shutter", 
    "A Tale of Two Sisters", "La Chinoise", "The Umbrellas of Cherbourg", "House",
    "Suspiria", "Blue Velvet"]
country = ["American", "American", "France", "Korean", "Thailand", "Korean", "France",
    "France", "Japan", "Italy", "American"]
runtime = [106, 119, 108, 120, 97, 115, 96, 91, 88, 98, 120]
year = [2023, 1941, 2017, 2017, 2004, 2003, 1967, 1964, 1977, 1977, 1986]
movie_dict = {"title" : titles, "country" : country, "runtime" : runtime, "year" : year}
df_movies = pd.DataFrame(movie_dict)


In [26]:
df_movies["country"] = df_movies["country"].replace("American", "USA")
df_movies["country"] = df_movies["country"].replace("Korean", "ROK")
df_movies["country"] = df_movies["country"].replace("Thailand", "THA")

2. Subset → group → aggregate → filter (one chained line)

From df_movies, subset rows with runtime >= 95.

Group by country and compute two stats:
average runtime, 2) number of titles (call it n_titles).
Then keep only countries with average runtime ≥ 100.

Return just country, avg_runtime, and n_titles.
Do this in one line using chaining.

In [16]:
high_runtime_country = (
    df_movies.query("runtime >= 95")
    .groupby("country", as_index = False)
    .agg(avg_runtime = ("runtime", "mean"),
         n_titles = ("title", "count"))
    .query("avg_runtime >= 100")
    [["country","avg_runtime","n_titles"]]
)

display(high_runtime_country)

,country,avg_runtime,n_titles
0,American,115.0,3
1,France,102.0,2
3,ROK,117.5,2


(3) Recode a numeric column with pd.cut

Using df_movies, recode year into a new column era with bins:
(-∞, 1960], (1960, 1990], (1990, 2009], (2009, ∞)
and labels: ["≤1960", "1961–1990", "1991–2009", "2010–onwards"].

Include the lowest value; use right-closed intervals.

In [ ]:
year_bins = [-np.inf, 1960, 1990, 2009, np.inf]
year_labels = ["before 1961", "1961-1990", "1991-2009", "2010 and onwards"]

df_movies["year_brackets"] = pd.cut(df_movies["year"],
                                    bins = year_bins,
                                    labels = year_labels,
                                    right = True,
                                    include_lowest = True
                                    )


(4) Merge two DataFrames (add exactly one column)

Left-merge df_movies (primary) with a subset of df_scores that contains only ["title", "imdb_score"], joining on "title".

Name the result movies_merge.

After merging, the primary DataFrame should gain only one new column: imdb_score.

In [23]:
titles = ["Past Lives", "Citizen Kane", "Revenge", "Okja", "Shutter", 
    "A Tale of Two Sisters", "La Chinoise", "The Umbrellas of Cherbourg", "House",
    "Suspiria", "Blue Velvet"]
scores = [7.8, 8.2, 6.4, 7.3, 7.0, 7.1, 6.9, 7.8, 7.2, 7.3, 7.7]
directors = ["Celine Song", "Orson Welles", "Coralie Fargeat", "Bong Joon Ho", 
    "Banjong Pisanthanakun", "Kim Jee-woon", "Jean-Luc Godard", "Jacques Demy", 
    "Nobuhiko Obayashi", "Dario Argento", "David Lynch"]

movie_dict2 = {"title" : titles, "imdb_score" : scores, "director" : directors}
df_scores = pd.DataFrame(movie_dict2)

scores_subset = df_scores[["title", "imdb_score"]] # remember two brackets here

movies_merge1 = pd.merge(df_movies,
                         scores_subset,
                         on = "title",
                         how = "left")

display(movies_merge1)

,title,country,runtime,year,year_brackets,imdb_score
0,Past Lives,American,106,2023,2010 and onwards,7.8
1,Citizen Kane,American,119,1941,before 1961,8.2
2,Revenge,France,108,2017,2010 and onwards,6.4
3,Okja,ROK,120,2017,2010 and onwards,7.3
4,Shutter,THA,97,2004,1991-2009,7.0
5,A Tale of Two Sisters,ROK,115,2003,1991-2009,7.1
6,La Chinoise,France,96,1967,1961-1990,6.9
7,The Umbrellas of Cherbourg,France,91,1964,1961-1990,7.8
8,House,Japan,88,1977,1961-1990,7.2
9,Suspiria,Italy,98,1977,1961-1990,7.3


(6) Query and pick “best per group”

Using movies_merge, subset to countries USA, France, Japan.

Within each country, select the highest-rated movie (use imdb_score; if there’s a tie, pick the oldest year).

Return a tidy DataFrame with columns: country, title, year, imdb_score.

In [28]:
country = (movies_merge1.query("country in ['USA','France','Japan']")
                        .sort_values(["country","imdb_score","year"], ascending=[True, False, True])
                        .drop_duplicates(subset="country", keep="first")[["country","title","year","imdb_score"]])

